# Reversión de tablas Medallion

El notebook consulta las seis tablas del proyecto y muestra su tipo y ubicación en Unity Catalog. La acción `review` presenta el inventario. La acción `drop` procesa las tablas administradas en orden Gold, Silver y Bronze.

Unity Catalog programa la limpieza de los archivos administrados después del periodo de recuperación. [Ciclo de vida del almacenamiento](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/object-storage-lifecycle)

In [ ]:
dbutils.widgets.text("catalog", "catalog_smartdata", "Catalog")
dbutils.widgets.dropdown("action", "review", ["review", "drop"], "Action")


In [ ]:
catalog = dbutils.widgets.get("catalog")
action = dbutils.widgets.get("action")

project_tables = [
    "bitcoin_gold.sentiment_summary",
    "bitcoin_gold.bitcoin_daily_metrics",
    "bitcoin_silver.bitcoin_daily",
    "bitcoin_silver.binance_minutes",
    "bitcoin_bronze.fear_greed_index",
    "bitcoin_bronze.binance_prices",
]

inventory = []
for table in project_tables:
    name = f"{catalog}.{table}"
    if not spark.catalog.tableExists(name):
        inventory.append((name, "AUSENTE", ""))
        continue

    description = spark.sql(
        "DESCRIBE TABLE EXTENDED IDENTIFIER(:name)",
        args={"name": name},
    )
    details = {row.col_name: row.data_type for row in description.collect()}
    inventory.append((name, details.get("Type", "DESCONOCIDO"), details.get("Location", "")))

display(spark.createDataFrame(inventory, ["tabla", "tipo", "ubicacion"]))

if action == "drop":
    unexpected = [name for name, kind, _ in inventory if kind not in ("MANAGED", "AUSENTE")]
    if unexpected:
        raise ValueError(f"Se encontraron tablas no administradas: {', '.join(unexpected)}")

    for name, kind, _ in inventory:
        if kind == "MANAGED":
            spark.sql("DROP TABLE IF EXISTS IDENTIFIER(:name)", args={"name": name})
            print(f"Eliminada: {name}")
